### Install and Import Libraries

In [ ]:
!pip install pandas
import pandas as pd

### Load and Inspect Raw Data

In [ ]:
# Load raw dataset from file path
df = pd.read_csv(r"Dataset\raw_dataset\Sales Dataset.csv")
# Preview first few rows to understand structure 
df.head()
# check data types, null values and overall structure
df.info()

### Checking for duplicates

In [ ]:
df.duplicated().any()

### Standardize column names

In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ","_")
    .str.replace("-","_")
    )

### Rename specific Column Names

In [ ]:
# Rename specific columns
df = df.rename(columns={
    "customername":"customer_name",
    "paymentmode":"payment_mode"
})
df.columns

### Fix Data Types

In [ ]:
# Convert order_date to datetime format
df["order_date"] = pd.to_datetime(df["order_date"])
# Convert year-month to proper datetime
df["year_month"] = pd.to_datetime(df["year_month"], format = "%Y-%m")
# Convert amount and profit to float
df["amount"] = df["amount"].astype(float)
df["profit"] = df["profit"].astype(float)
df.info()

### Create Dimension Table

In [ ]:
try:
    product_dim = df[["category","sub_category"]].copy().drop_duplicates().reset_index(drop=True)
    product_dim["product_id"] = product_dim.index + 1

    customer_dim = df[["customer_name"]].copy().drop_duplicates().reset_index(drop=True)
    customer_dim["customer_id"] = customer_dim.index + 1

    payment_dim = df[["payment_mode"]].copy().drop_duplicates().reset_index(drop=True)
    payment_dim["payment_id"] = payment_dim.index + 1

    location_dim = df[["state","city"]].copy().drop_duplicates().reset_index(drop=True)
    location_dim["location_id"] = location_dim.index + 1

    date_dim = df[["order_date"]].copy().drop_duplicates().reset_index(drop=True)
    date_dim["date_id"] = date_dim.index + 1

# Extract date attributes
    date_dim["year"] = date_dim["order_date"].dt.year
    date_dim["day_name"] = date_dim["order_date"].dt.day_name()
    date_dim["month_name"] = date_dim["order_date"].dt.month_name()
    date_dim["quarter"] = date_dim["order_date"].dt.quarter
    date_dim.head()

    print("All Dimension tables created successfully")
except Exception as e:
    print("Unable to create dimension tables",e)


### Merge Dimension IDs Back to Main Data

In [ ]:

df = df.merge(product_dim, on = ["category","sub_category"])
df = df.merge(customer_dim, on = "customer_name")
df = df.merge(payment_dim, on = "payment_mode")
df = df.merge(location_dim, on = ["state","city"])
df = df.merge(date_dim,on= "order_date") 

### Create Fact Table (Orders)

In [ ]:
try:
    orders_fact = df[["order_id","customer_id","product_id","date_id","location_id","payment_id","amount","quantity","profit",
    ]].copy()

    orders_fact["order_items_id"] = range(1,len(orders_fact) + 1)
    orders_fact.head(3)

    print("Fact table successfully created")
except Exception as e:
    print("Unable to create the Fact table",e)

### Export Cleaned Data

In [ ]:
customer_dim.to_csv(r"Dataset\clean_dataset\customers.csv",index = False)
product_dim.to_csv(r"Dataset\clean_dataset\products.csv",index = False)
location_dim.to_csv(r"Dataset\clean_dataset\locations.csv",index = False)
payment_dim.to_csv(r"Dataset\clean_dataset\payments.csv",index = False)
orders_fact.to_csv(r"Dataset\clean_dataset\orders.csv",index = False)
date_dim.to_csv(r"Dataset\clean_dataset\dates.csv",index = False)



### viewing the columns

In [ ]:
df.columns

### Transactional tables

In [ ]:
try:
    payments = df[["payment_id","payment_mode"]].copy().drop_duplicates().reset_index(drop=True)

    customers= df[["customer_id","customer_name"]].copy().drop_duplicates().reset_index(drop=True)

    products = df[["category","sub_category","amount"]].copy().drop_duplicates(subset=["sub_category"]).reset_index(drop=True)
    products['product_id']= product_dim.index + 1


    locations = df[["location_id","city","state"]].copy().drop_duplicates().reset_index(drop=True)


    orders = df[["order_id","payment_id","customer_id","product_id","location_id","order_date","quantity","profit"]].copy()
    orders["order_items_id"] = range(1,len(orders) + 1)
    orders = orders[["order_items_id","order_id","payment_id","customer_id","product_id","location_id","order_date","quantity","profit"]].copy()
    orders.head(3)

    print("Transactions table exported")
except Exception as e:
    print("Unable to create transactional tables")

### Export the Transactional tables

In [ ]:
payments.to_csv(r"Dataset\transactions_data\payments.csv", index=False)
customers.to_csv(r"Dataset\transactions_data\customers.csv",index=False)
products.to_csv(r"Dataset\transactions_data\products.csv",index=False)
locations.to_csv(r"Dataset\transactions_data\locations.csv",index=False)
orders.to_csv(r"Dataset\transactions_data\orders.csv",index=False)

In [ ]:
df.info()